# V6.2B — Deduplicate TRAIN+VAL, Rebalance, Keep TEST Exactly Fixed

This notebook fixes the failure:

```text
Exact duplicate overlap train-val: 35
AssertionError
```

## What happened

Those duplicate images were already present **inside the old TRAIN+VAL pool**.
V6.1 only proved that exact duplicates did not cross the old split boundaries.
When TRAIN+VAL were re-stratified, two copies of the same image could be assigned
to opposite sides, which exposed the hidden duplication.

## Correct policy

1. Keep TEST **byte-for-byte unchanged**.
2. Combine old TRAIN + VAL.
3. Group by image SHA256.
4. For duplicate groups:
   - verify their normalized labels agree;
   - keep one deterministic representative;
   - quarantine redundant copies.
5. Re-split the **deduplicated** TRAIN+VAL pool using multilabel stratification.
6. Keep Validation at **2,263 images**.
7. Train size becomes slightly smaller because redundant copies are removed.
8. Verify:
   - no exact SHA overlap train↔val;
   - no exact SHA overlap train/val↔test;
   - TEST hash set is exactly unchanged;
   - bird_nest is no longer starved in validation.
9. Freeze as a new dataset tag.

No model training happens here.


In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, hashlib, zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone

W = Path("/kaggle/working")
TMP = Path("/tmp/v6p2b")
BASE_EXTRACT = TMP/"base"
OVERLAY_EXTRACT = TMP/"overlay"
OLD_RUNTIME = TMP/"v6p1_runtime"
NEW_RUNTIME = TMP/"v6p2b_runtime"
REPORTS = W/"v6p2b_freeze_records"

for p in [TMP, BASE_EXTRACT, OVERLAY_EXTRACT, REPORTS]:
    p.mkdir(parents=True, exist_ok=True)

PARENT_TAG = "v6p1_5d0b1149891c"
BASE_TAG = "56b7ea194ee5"
OVERLAY_TAG = "ee2d3c4497fd"

DATA_ROOT = "stash://bunpmc/projects/yolov11sdi/datasets"
PARENT_ROOT = f"{DATA_ROOT}/v6p1/{PARENT_TAG}"

BASE_STASH = f"{DATA_ROOT}/curated_v6_rs1280_{BASE_TAG}.tar"
OVERLAY_STASH = f"{PARENT_ROOT}/v6p1_fotl_overlay_{OVERLAY_TAG}.tar.gz"
FREEZE_STASH = f"{PARENT_ROOT}/v6p1.freeze.json"

NAMES = [
    "insulator",
    "insulator_broken",
    "insulator_flashover",
    "bird_nest",
    "foreign_object",
    "broken_strand",
]

OLD_SPLITS = {"train":17655, "val":2263, "test":2012}
TARGET_VAL_IMAGES = 2263
SEED = 42
STRICT_DUPLICATE_LABEL_CONFLICTS = True
UPLOAD_TO_CAMBER = True
CAMBER_ROOT = f"{DATA_ROOT}/v6p2b"

IMG_EXTS = {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}


In [ ]:
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pyyaml", "pandas", "iterative-stratification"
])

import numpy as np
import pandas as pd
import yaml
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

def images_under(root):
    return sorted(
        p for p in Path(root).rglob("*")
        if p.is_file() and p.suffix.lower() in IMG_EXTS
    )

def sha256_file(path, chunk=4*1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(chunk),b""):
            h.update(b)
    return h.hexdigest()

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def normalized_label_signature(label_path):
    p=Path(label_path)
    rows=[]
    if p.exists():
        for line in p.read_text(errors="ignore").splitlines():
            if not line.strip():
                continue
            parts=line.split()
            if len(parts) < 5:
                raise RuntimeError(f"Malformed label: {p}: {line}")
            cid=int(float(parts[0]))
            vals=tuple(round(float(x),6) for x in parts[1:5])
            rows.append((cid,*vals))
    return tuple(sorted(rows))


## 1. Restore exact frozen V6.1 artifacts

In [ ]:
from kaggle_secrets import UserSecretsClient

key=UserSecretsClient().get_secret("CAMBER_API_KEY")
if not key:
    raise RuntimeError("Missing CAMBER_API_KEY")

os.environ["CAMBER_API_KEY"]=key

if shutil.which("camber") is None and not (Path.home()/".camber/bin/camber").exists():
    subprocess.check_call("curl -sL https://cli.cambercloud.com/install-v2.sh | bash",shell=True)

os.environ["PATH"]=f"{Path.home()}/.camber/bin:{Path.home()}/.local/bin:"+os.environ.get("PATH","")
CAMBER=shutil.which("camber") or str(Path.home()/".camber/bin/camber")

def stash_cp(src,dst,required=True):
    dst=Path(dst); dst.parent.mkdir(parents=True,exist_ok=True)
    r=subprocess.run(
        [CAMBER,"stash","cp",str(src),str(dst)],
        text=True,capture_output=True,env=os.environ.copy()
    )
    ok=r.returncode==0 and dst.exists()
    if required and not ok:
        raise RuntimeError(r.stderr[-1800:] or r.stdout[-1800:])
    return ok

def stash_put(src,dst):
    src=Path(src)
    r=subprocess.run(
        [CAMBER,"stash","cp",str(src),str(dst)],
        text=True,capture_output=True,env=os.environ.copy()
    )
    if r.returncode!=0:
        raise RuntimeError(r.stderr[-1800:] or r.stdout[-1800:])
    print("Uploaded:",dst)

parent_freeze=TMP/"v6p1.freeze.json"
stash_cp(FREEZE_STASH,parent_freeze)
parent=json.loads(parent_freeze.read_text())

assert parent["final_tag"]==PARENT_TAG
assert parent["freeze_ready"] is True
assert parent["final_split_counts"]==OLD_SPLITS

base_tar=TMP/f"base_{BASE_TAG}.tar"
overlay_tar=TMP/f"overlay_{OVERLAY_TAG}.tar.gz"

if not any(BASE_EXTRACT.rglob("data.yaml")):
    stash_cp(BASE_STASH,base_tar)
    subprocess.check_call(["tar","-xf",str(base_tar),"-C",str(BASE_EXTRACT)])
    base_tar.unlink(missing_ok=True)

if not any(OVERLAY_EXTRACT.rglob("data.yaml")):
    stash_cp(OVERLAY_STASH,overlay_tar)
    subprocess.check_call(["tar","-xzf",str(overlay_tar),"-C",str(OVERLAY_EXTRACT)])
    overlay_tar.unlink(missing_ok=True)

def normalize_names(names):
    if isinstance(names,dict):
        return [names[k] for k in sorted(names,key=lambda x:int(x))]
    return list(names)

def find_root(root):
    for yp in Path(root).rglob("data.yaml"):
        try:
            cfg=yaml.safe_load(yp.read_text())
            if normalize_names(cfg.get("names",[]))==NAMES:
                return yp.parent
        except Exception:
            pass
    raise RuntimeError(f"No matching dataset under {root}")

BASE=find_root(BASE_EXTRACT)
OVERLAY=find_root(OVERLAY_EXTRACT)

print("BASE:",BASE)
print("OVERLAY:",OVERLAY)


## 2. Reconstruct exact V6.1 union

In [ ]:
if OLD_RUNTIME.exists():
    shutil.rmtree(OLD_RUNTIME)

for sp in ["train","val","test"]:
    for src in ["base","fotl"]:
        (OLD_RUNTIME/"images"/sp/src).mkdir(parents=True,exist_ok=True)
        (OLD_RUNTIME/"labels"/sp/src).mkdir(parents=True,exist_ok=True)

def link_source(src_root, split, source_name, runtime_root):
    iroot=src_root/"images"/split
    lroot=src_root/"labels"/split

    for ip in images_under(iroot):
        rel=ip.relative_to(iroot)
        lp=lroot/rel.with_suffix(".txt")

        dip=runtime_root/"images"/split/source_name/rel
        dlp=runtime_root/"labels"/split/source_name/rel.with_suffix(".txt")

        dip.parent.mkdir(parents=True,exist_ok=True)
        dlp.parent.mkdir(parents=True,exist_ok=True)

        os.symlink(ip,dip)
        if lp.exists():
            os.symlink(lp,dlp)
        else:
            dlp.write_text("")

for sp in ["train","val","test"]:
    link_source(BASE,sp,"base",OLD_RUNTIME)
    link_source(OVERLAY,sp,"fotl",OLD_RUNTIME)

old_counts={
    sp:len(images_under(OLD_RUNTIME/"images"/sp))
    for sp in ["train","val","test"]
}
print(old_counts)
assert old_counts==OLD_SPLITS


## 3. Build image records and lock TEST

In [ ]:
records=[]

for sp in ["train","val","test"]:
    iroot=OLD_RUNTIME/"images"/sp
    lroot=OLD_RUNTIME/"labels"/sp

    for ip in images_under(iroot):
        rel=ip.relative_to(iroot)
        source=rel.parts[0]
        lp=lroot/rel.with_suffix(".txt")

        box_counts=[0]*len(NAMES)
        if lp.exists():
            for line in lp.read_text(errors="ignore").splitlines():
                if not line.strip():
                    continue
                parts=line.split()
                cid=int(float(parts[0]))
                vals=list(map(float,parts[1:5]))
                assert 0 <= cid < len(NAMES), (lp,cid)
                assert all(0 <= x <= 1 for x in vals), (lp,vals)
                box_counts[cid]+=1

        records.append({
            "old_split":sp,
            "source":source,
            "relpath":rel.as_posix(),
            "image_path":str(ip),
            "label_path":str(lp),
            "sha256":sha256_file(ip),
            "label_signature":normalized_label_signature(lp),
            "class_presence":[int(x>0) for x in box_counts],
            "box_counts":box_counts,
        })

test_records=[r for r in records if r["old_split"]=="test"]
pool_records=[r for r in records if r["old_split"] in ("train","val")]

OLD_TEST_HASHES=sorted(r["sha256"] for r in test_records)

assert len(test_records)==OLD_SPLITS["test"]
assert len(pool_records)==OLD_SPLITS["train"]+OLD_SPLITS["val"]

print("TEST locked:",len(test_records))
print("TRAIN+VAL raw pool:",len(pool_records))


## 4. Deduplicate TRAIN+VAL before splitting

For each SHA256 group:
- one deterministic representative is kept;
- redundant copies are quarantined;
- conflicting annotations hard-stop the notebook.


In [ ]:
by_hash=defaultdict(list)
for r in pool_records:
    by_hash[r["sha256"]].append(r)

unique_pool=[]
quarantine=[]
conflicts=[]

for h,grp in sorted(by_hash.items()):
    sigs=defaultdict(list)
    for r in grp:
        sigs[r["label_signature"]].append(r)

    if len(sigs)>1:
        conflicts.append({
            "sha256":h,
            "n_copies":len(grp),
            "paths":" | ".join(
                f"{r['old_split']}:{r['source']}:{r['relpath']}"
                for r in grp
            ),
            "n_distinct_label_signatures":len(sigs),
            "box_counts":" | ".join(str(r["box_counts"]) for r in grp),
        })

    rep=sorted(
        grp,
        key=lambda r:(0 if r["source"]=="base" else 1, r["relpath"])
    )[0]
    unique_pool.append(rep)

    for r in grp:
        if r is rep:
            continue
        quarantine.append({
            "sha256":h,
            "kept_source":rep["source"],
            "kept_old_split":rep["old_split"],
            "kept_relpath":rep["relpath"],
            "removed_source":r["source"],
            "removed_old_split":r["old_split"],
            "removed_relpath":r["relpath"],
            "same_label_signature":r["label_signature"]==rep["label_signature"],
        })

conflict_df=pd.DataFrame(conflicts)
quarantine_df=pd.DataFrame(quarantine)

conflict_df.to_csv(REPORTS/"duplicate_annotation_conflicts.csv",index=False)
quarantine_df.to_csv(REPORTS/"trainval_exact_duplicate_quarantine.csv",index=False)

print("TRAIN+VAL raw images:",len(pool_records))
print("Unique SHA256 images:",len(unique_pool))
print("Redundant copies removed:",len(quarantine))
print("Duplicate hash groups:",sum(1 for g in by_hash.values() if len(g)>1))
print("Conflicting annotation groups:",len(conflicts))

if conflicts and STRICT_DUPLICATE_LABEL_CONFLICTS:
    raise RuntimeError(
        f"Found {len(conflicts)} exact-image groups with conflicting annotations. "
        "Review duplicate_annotation_conflicts.csv before freezing."
    )

assert len(unique_pool)+len(quarantine)==len(pool_records)


## 5. Confirm TEST still has no overlap with deduplicated TRAIN+VAL pool

In [ ]:
pool_hash=set(r["sha256"] for r in unique_pool)
test_hash=set(r["sha256"] for r in test_records)

cross_pool_test=pool_hash & test_hash
print("Exact overlap deduped TRAIN+VAL pool ↔ TEST:",len(cross_pool_test))
assert not cross_pool_test

print("✅ TEST remains independent by exact SHA256.")


## 6. Re-split deduplicated TRAIN+VAL

Validation remains exactly 2,263 images.

The stratification vector contains:
- six class-presence indicators;
- source_base;
- source_fotl;
- an extra bird_nest-presence indicator so the previously starved class receives additional weight.


In [ ]:
Y=[]

bird_id=NAMES.index("bird_nest")

for r in unique_pool:
    class_bits=list(r["class_presence"])
    source_bits=[
        int(r["source"]=="base"),
        int(r["source"]=="fotl"),
    ]
    bird_extra=[r["class_presence"][bird_id]]
    Y.append(class_bits+source_bits+bird_extra)

Y=np.asarray(Y,dtype=np.int64)
X=np.arange(len(unique_pool)).reshape(-1,1)

assert len(unique_pool) > TARGET_VAL_IMAGES

splitter=MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=TARGET_VAL_IMAGES,
    random_state=SEED,
)

idx_train,idx_val=next(splitter.split(X,Y))

new_train=[unique_pool[int(i)] for i in idx_train]
new_val=[unique_pool[int(i)] for i in idx_val]
new_test=test_records

print("New train:",len(new_train))
print("New val:",len(new_val))
print("Fixed test:",len(new_test))
print("New total:",len(new_train)+len(new_val)+len(new_test))


## 7. Zero-leakage assertions

In [ ]:
train_hash=set(r["sha256"] for r in new_train)
val_hash=set(r["sha256"] for r in new_val)
new_test_hash=set(r["sha256"] for r in new_test)

cross_tv=train_hash & val_hash
cross_tt=train_hash & new_test_hash
cross_vt=val_hash & new_test_hash

print("Exact duplicate overlap train-val:",len(cross_tv))
print("Exact duplicate overlap train-test:",len(cross_tt))
print("Exact duplicate overlap val-test:",len(cross_vt))

assert not cross_tv
assert not cross_tt
assert not cross_vt

NEW_TEST_HASHES=sorted(r["sha256"] for r in new_test)
assert NEW_TEST_HASHES==OLD_TEST_HASHES

print("✅ TRAIN↔VAL exact leakage = 0")
print("✅ TRAIN/VAL↔TEST exact leakage = 0")
print("✅ TEST SHA256 set is exactly unchanged")


## 8. Distribution audit

In [ ]:
def summarize(recs):
    img=Counter()
    box=Counter()
    for r in recs:
        for cid,p in enumerate(r["class_presence"]):
            if p:
                img[cid]+=1
        for cid,n in enumerate(r["box_counts"]):
            box[cid]+=n
    return img,box

old_summary={
    sp:summarize([r for r in records if r["old_split"]==sp])
    for sp in ["train","val","test"]
}

new_summary={
    "train":summarize(new_train),
    "val":summarize(new_val),
    "test":summarize(new_test),
}

rows=[]
for cid,name in enumerate(NAMES):
    for sp in ["train","val","test"]:
        oi,ob=old_summary[sp]
        ni,nb=new_summary[sp]
        rows.append({
            "class_id":cid,
            "class_name":name,
            "split":sp,
            "v6p1_images":oi[cid],
            "v6p2b_images":ni[cid],
            "v6p1_boxes":ob[cid],
            "v6p2b_boxes":nb[cid],
        })

cmp=pd.DataFrame(rows)
cmp.to_csv(REPORTS/"v6p1_vs_v6p2b_distribution.csv",index=False)
display(cmp)
display(cmp[cmp.class_name=="bird_nest"])


In [ ]:
balance_rows=[]

for cid,name in enumerate(NAMES):
    train_i=new_summary["train"][0][cid]
    val_i=new_summary["val"][0][cid]
    train_b=new_summary["train"][1][cid]
    val_b=new_summary["val"][1][cid]

    pool_i=train_i+val_i
    pool_b=train_b+val_b

    balance_rows.append({
        "class_id":cid,
        "class_name":name,
        "train_images":train_i,
        "val_images":val_i,
        "val_image_ratio_within_trainval":val_i/pool_i if pool_i else 0,
        "train_boxes":train_b,
        "val_boxes":val_b,
        "val_box_ratio_within_trainval":val_b/pool_b if pool_b else 0,
    })

balance=pd.DataFrame(balance_rows)
balance.to_csv(REPORTS/"v6p2b_trainval_balance.csv",index=False)
display(balance)

bird_row=balance[balance.class_name=="bird_nest"].iloc[0]

assert bird_row["val_images"] >= 500, (
    f"bird_nest val images still too small: {bird_row['val_images']}"
)
assert bird_row["val_boxes"] >= 500, (
    f"bird_nest val boxes still too small: {bird_row['val_boxes']}"
)

print("✅ bird_nest validation support is now materially larger.")


## 9. Build runtime split by symlink

In [ ]:
if NEW_RUNTIME.exists():
    shutil.rmtree(NEW_RUNTIME)

for sp in ["train","val","test"]:
    (NEW_RUNTIME/"images"/sp).mkdir(parents=True,exist_ok=True)
    (NEW_RUNTIME/"labels"/sp).mkdir(parents=True,exist_ok=True)

def link_record(r,sp):
    rel=Path(r["relpath"])
    dip=NEW_RUNTIME/"images"/sp/rel
    dlp=NEW_RUNTIME/"labels"/sp/rel.with_suffix(".txt")

    dip.parent.mkdir(parents=True,exist_ok=True)
    dlp.parent.mkdir(parents=True,exist_ok=True)

    os.symlink(r["image_path"],dip)

    lp=Path(r["label_path"])
    if lp.exists():
        os.symlink(lp,dlp)
    else:
        dlp.write_text("")

for r in new_train: link_record(r,"train")
for r in new_val: link_record(r,"val")
for r in new_test: link_record(r,"test")

new_counts={
    sp:len(images_under(NEW_RUNTIME/"images"/sp))
    for sp in ["train","val","test"]
}
print("Final V6.2B split counts:",new_counts)

(NEW_RUNTIME/"data.yaml").write_text(yaml.safe_dump({
    "path":str(NEW_RUNTIME),
    "train":"images/train",
    "val":"images/val",
    "test":"images/test",
    "names":NAMES,
},sort_keys=False))


## 10. Freeze split manifest

In [ ]:
manifest=[]

for sp,recs in [("train",new_train),("val",new_val),("test",new_test)]:
    for r in recs:
        manifest.append({
            "split":sp,
            "source":r["source"],
            "relpath":r["relpath"],
            "sha256":r["sha256"],
            "previous_v6p1_split":r["old_split"],
        })

manifest_df=pd.DataFrame(manifest).sort_values(["split","source","relpath"])
manifest_path=REPORTS/"v6p2b_split_manifest.csv"
manifest_df.to_csv(manifest_path,index=False)

h=hashlib.sha256()
h.update(PARENT_TAG.encode())
h.update(json.dumps(NAMES).encode())
for row in manifest_df.to_dict("records"):
    h.update(json.dumps(row,sort_keys=True).encode())

SPLIT_SHA=h.hexdigest()
FINAL_TAG=f"v6p2b_{SPLIT_SHA[:12]}"

freeze={
    "schema_version":"v6p2b-dedup-resplit-1",
    "created_utc":utc_now(),
    "dataset_name":"V6.2B",
    "final_tag":FINAL_TAG,
    "split_sha256":SPLIT_SHA,
    "parent_dataset_tag":PARENT_TAG,
    "base_tag":BASE_TAG,
    "overlay_tag":OVERLAY_TAG,
    "taxonomy":NAMES,
    "dedup_policy":{
        "scope":"V6.1 TRAIN+VAL only",
        "method":"exact image SHA256",
        "raw_trainval_images":len(pool_records),
        "unique_trainval_images":len(unique_pool),
        "redundant_copies_removed":len(quarantine),
        "duplicate_annotation_conflict_groups":len(conflicts),
    },
    "split_policy":{
        "test":"exact SHA256 set preserved from V6.1",
        "validation_images":len(new_val),
        "training_images":len(new_train),
        "method":"MultilabelStratifiedShuffleSplit",
        "features":"6 class-presence + source + extra bird_nest presence",
        "seed":SEED,
    },
    "final_split_counts":new_counts,
    "integrity":{
        "train_val_exact_overlap":len(cross_tv),
        "train_test_exact_overlap":len(cross_tt),
        "val_test_exact_overlap":len(cross_vt),
        "test_hash_set_unchanged":NEW_TEST_HASHES==OLD_TEST_HASHES,
    },
    "freeze_ready":True,
}

freeze_path=REPORTS/"v6p2b.freeze.json"
freeze_path.write_text(json.dumps(freeze,indent=2))

print(json.dumps(freeze,indent=2))


## 11. Final report + Camber upload

In [ ]:
final_rows=[]

for cid,name in enumerate(NAMES):
    row={"class":name}
    total_boxes=0
    for sp in ["train","val","test"]:
        row[f"{sp}_images"]=new_summary[sp][0][cid]
        row[f"{sp}_boxes"]=new_summary[sp][1][cid]
        total_boxes += new_summary[sp][1][cid]
    row["total_boxes"]=total_boxes
    final_rows.append(row)

final_df=pd.DataFrame(final_rows)
final_df.to_csv(REPORTS/"v6p2b_final_distribution.csv",index=False)
display(final_df)

lines=[
    "# V6.2B Dataset Report",
    "",
    f"- Final tag: `{FINAL_TAG}`",
    f"- Parent: `{PARENT_TAG}`",
    f"- Train: {len(new_train)}",
    f"- Validation: {len(new_val)}",
    f"- Test: {len(new_test)}",
    f"- Redundant TRAIN+VAL exact copies removed: {len(quarantine)}",
    f"- TEST unchanged: {NEW_TEST_HASHES==OLD_TEST_HASHES}",
    "",
    "## Final distribution",
    "",
    final_df.to_markdown(index=False),
    "",
    "## Methodological correction",
    "",
    "V6.1 passed cross-split integrity checks but contained exact duplicate copies "
    "inside the combined training/validation pool. Re-stratification exposed those "
    "duplicates across the new TRAIN/VAL boundary. V6.2B therefore removes redundant "
    "exact copies from TRAIN+VAL before re-splitting, while preserving the TEST set "
    "exactly as frozen in V6.1.",
]

report_path=REPORTS/"V6P2B_DATASET_REPORT.md"
report_path.write_text("\n".join(lines))

records_zip=W/f"{FINAL_TAG}_freeze_records.zip"
with zipfile.ZipFile(records_zip,"w",zipfile.ZIP_DEFLATED) as z:
    for p in REPORTS.rglob("*"):
        if p.is_file():
            z.write(p,p.relative_to(REPORTS.parent))

if UPLOAD_TO_CAMBER:
    remote=f"{CAMBER_ROOT}/{FINAL_TAG}"
    for p in [
        freeze_path,
        manifest_path,
        REPORTS/"trainval_exact_duplicate_quarantine.csv",
        REPORTS/"duplicate_annotation_conflicts.csv",
        REPORTS/"v6p1_vs_v6p2b_distribution.csv",
        REPORTS/"v6p2b_trainval_balance.csv",
        REPORTS/"v6p2b_final_distribution.csv",
        report_path,
        records_zip,
    ]:
        if p.exists():
            stash_put(p,f"{remote}/{p.name}")

    print("✅ FINAL DATASET TAG:",FINAL_TAG)
    print("✅ TEST preserved exactly from:",PARENT_TAG)
    print("✅ CAMBER ROOT:",remote)
else:
    print("FINAL DATASET TAG:",FINAL_TAG)


# Next step

Only after this notebook prints:

```text
✅ TRAIN↔VAL exact leakage = 0
✅ TEST SHA256 set is exactly unchanged
✅ bird_nest validation support is now materially larger
✅ FINAL DATASET TAG: v6p2b_...
```

should baseline training resume.

Use the resulting `v6p2b_...` tag for every YOLO / RT-DETR experiment.
